<a href="https://colab.research.google.com/github/Eleonwra/elcardiocc-baseline-ner/blob/main/preprocessing/split_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
data = []
with open('train_dataset.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))

In [ ]:
result_list = []

for entry in data:
    patient_id = entry['id']
    text = entry['text']
    mentions = []
    for label in entry['annotations']:
        label_values = list(label.values())
        mentions.append(label_values)
    result_list.append({
            'patient_id': patient_id,
            'label': mentions,
            'text': text
        })

In [ ]:
import pandas as pd
result_df = []

for entry in data:
    patient_id = entry['id']
    text = entry['text']

    for annotation in entry['annotations']:
        result_df.append({
            'patient_id': patient_id,
            'start': annotation['start'],
            'end': annotation['end'],
            'code': annotation['code'],
            'mention': annotation['mention']
        })
result_df = pd.DataFrame(result_df)

       patient_id  start   end code                mention
0               3    220   226  I21                 NSTEMI
1               3    530   544  Y84         στεφανιογραφία
2               3    549   563  Z95         αγγειοπλαστική
3               3   1003  1017  Y84         Στεφανιογραφία
4               3   1020  1034  Z95         Αγγειοπλαστική
...           ...    ...   ...  ...                    ...
10163        6559    425   428  I21                    ΟΕΜ
10164        6559    555   568  E78          Δυσλιπιδαιμία
10165        6559    655   669  I30         περικαρδίτιδος
10166        6559    730   751  I30  οξείας περικαρδίτιδας
10167        6559   1037  1054  R07      Προκαρδίου Άλγους

[10168 rows x 5 columns]


In [ ]:
print(f"Total Annotations in the Dataset: {len(result_df)}")

patient_stats = result_df.groupby('patient_id').size().agg(['count', 'min', 'max', 'mean'])

print("Number of Patients/ EHR's:", patient_stats['count'])
print("Min Annotations per Patient:", patient_stats['min'])
print("Max Annotations per Patient:", patient_stats['max'])
print("Average Annotations per Patient:", patient_stats['mean'])

Total Annotations in the Dataset: 10168
Number of Patients/ EHR's: 1000.0
Min Annotations per Patient: 1.0
Max Annotations per Patient: 33.0
Average Annotations per Patient: 10.168


In [ ]:
j = 0
annotations_new = []
b_entity_count = 0
i_entity_count = 0
for i, document_data in enumerate(result_list):
    patient_id = document_data["patient_id"]
    text = document_data["text"]
    recognized_entities = []
    previous_end = -1
    for annotation in result_list[i]["label"]:
        start = annotation[0]
        end = annotation[1]
        icd_code = annotation[2]
        text1 = text
        name = text1[start:end]
        words = name.split()
        current_start = start
        for idx, word in enumerate(words):
            if idx == 0:
                type = "B-ENTITY"
                recognized_entities.append({
                "start": start,
                "end": start + len(word),
                "type": type,
                'word_name': word,
                'icd_code': icd_code })
                current_start = start + len(word)
                b_entity_count = b_entity_count + 1
        if current_start < end:
          type = "I-ENTITY"
          recognized_entities.append({
          "start": current_start,
          "end": end,
          "type": type,
          'word_name': word,
          'icd_code': icd_code })
          i_entity_count = i_entity_count + 1
          current_start = current_start + len(word)

    document_entry = {
        "patient_id": patient_id,
        "text": text,
        "recognized_entities": recognized_entities
    }
    annotations_new.append(document_entry)
print("Number of B entities:", b_entity_count)
print("Number of I entities:", i_entity_count)

Number of B entities: 10168
Number of I entities: 5557


In [ ]:
%pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which 

In [ ]:
from datasets import Dataset, Features, Sequence, Value, ClassLabel
tokens_list = []
ner_tags_list = []
patient_id_list = []
for j,dat in enumerate(annotations_new):
    tokens = (list(dat["text"]))
    ner_tags = ["O"] * len(tokens)
    for ent in annotations_new[j]["recognized_entities"]:
        for i in range(ent['start'], ent['end']):
            ner_tags[i] = ent['type']
    tokens_list.append(tokens)
    ner_tags_list.append(ner_tags)
    patient_id_list.append(dat['patient_id'])

features = Features({
    "tokens": Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
    "ner_tags": Sequence(feature=ClassLabel(names=["O", "B-ENTITY",'I-ENTITY'], id=None), length=-1, id=None),
    "patient_id": Value(dtype='int64', id=None)
})
ds = Dataset.from_dict(
    {"tokens": tokens_list, "ner_tags": ner_tags_list,"patient_id": patient_id_list},
    features=features
)

In [ ]:
from sklearn.model_selection import train_test_split
random_seed = 42
total_size = len(ds)
train_size = int(total_size * 0.8)
val_size = total_size-train_size

print(f"Total Size: {total_size}")
print(f"Train Size: {train_size}")
print(f"Validation Size: {val_size}")

patient_ids = ds['patient_id']
indices = range(len(patient_ids))

patient_ids = ds['patient_id']
patient_ids_train, patient_ids_val = train_test_split(patient_ids, test_size=val_size, random_state=random_seed)

ds_train_data = [obj for obj in ds if obj['patient_id'] in patient_ids_train]
ds_val_data = [obj for obj in ds if obj['patient_id'] in patient_ids_val]

ds_train = Dataset.from_dict({"tokens": [obj['tokens'] for obj in ds_train_data],
                              "ner_tags": [obj['ner_tags'] for obj in ds_train_data],
                              "patient_id": [obj['patient_id'] for obj in ds_train_data]})
ds_val = Dataset.from_dict({"tokens": [obj['tokens'] for obj in ds_val_data],
                            "ner_tags": [obj['ner_tags'] for obj in ds_val_data],
                            "patient_id": [obj['patient_id'] for obj in ds_val_data]})

final_dataset = {
    'train': ds_train,
    'validation': ds_val
}
print('Final dataset structure:')
final_dataset

Total Size: 1000
Train Size: 800
Validation Size: 200
Final dataset structure:


{'train': Dataset({
     features: ['tokens', 'ner_tags', 'patient_id'],
     num_rows: 800
 }),
 'validation': Dataset({
     features: ['tokens', 'ner_tags', 'patient_id'],
     num_rows: 200
 })}

In [ ]:
import pickle
with open('final_dataset.pickle', 'wb') as output:
    pickle.dump(final_dataset, output)